# Geographic Expansion & Site Selection

Build a **grounded, self-verifying** agent that tells a payments processor **where to expand** — ranking metro × merchant-category cells by **white-space opportunity**: strong category demand (real US Census retail sales) meeting low merchant coverage (synthetic processor book).

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`SITE_SELECTION_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** (top opportunities) |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Prereq:** run `sql/setup.sql` (fully synthetic, no listing), then optionally `sql/setup_census.sql` (real Census demand, synthetic geography — `mode = hybrid`). Use the role `SNOWPACK_SITE_SELECTION_ROLE`, which setup creates and which owns `SNOWPACK_SITE_SELECTION.CORE`.

> Geography (metros, population, income) is **synthetic** — real demographics (Census ACS) and POI (Overture) are paid Marketplace add-ons.

In [ ]:
SCHEMA = "SNOWPACK_SITE_SELECTION.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.SITE_SELECTION_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`SITE_SELECTION_BENCHMARK` = one row per **metro × category**, with penetration, white-space volume, real category demand growth, and a computed **opportunity_score** (demand growth × coverage gap).

In [ ]:
session.sql(f"""
  SELECT metro_name, state, category_name,
         ROUND(penetration_pct*100,1) AS penetration_pct,
         ROUND(white_space_volume/1e6,1) AS white_space_m,
         category_demand_growth_pct AS demand_growth_pct,
         opportunity_score
  FROM {SCHEMA}.SITE_SELECTION_BENCHMARK
  ORDER BY opportunity_score DESC
  LIMIT 12
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`SITE_SELECTION_SV` lets Analyst answer NL questions like *"which regions have the most white-space opportunity?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English.

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.SITE_SELECTION_SV
      METRICS avg_opportunity_score, total_white_space_usd, avg_penetration_pct
      DIMENSIONS region, category_name)
  ORDER BY avg_opportunity_score DESC
""").to_pandas()

## Step 3 — Verified fact pack

The top-12 metro × category opportunities — the agent's only allowed numbers.

In [ ]:
fact_pack = session.sql(f"""
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'metro', metro_name, 'state', state, 'region', region, 'category', category_name,
           'penetration_pct', penetration_pct, 'opportunity_score', opportunity_score,
           'white_space_usd', white_space_volume, 'merchants_per_100k', merchants_per_100k,
           'demand_growth_pct', category_demand_growth_pct))
  FROM (
    SELECT * FROM {SCHEMA}.SITE_SELECTION_BENCHMARK
    ORDER BY opportunity_score DESC
    LIMIT 12)
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a payments expansion strategist. Write a concise site-selection note using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; opportunity_score blends category demand growth with coverage gap "
    "(higher = stronger expansion target); penetration_pct is a ratio (0-1); white_space_usd is USD, express in $B/$M; "
    "no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (top expansion targets), a table (metro | category | penetration | opportunity | white space), then 3 bullets, ending with '## Data Coverage' (source + note)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.SITE_SELECTION_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Expansion white-space ranking", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.SITE_SELECTION_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified site-selection loop. `sql/setup.sql` already created a **`SITE_SELECTION_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which metros and categories have the most white space?"*).

**Next:** swap `SITE_SELECTION_METRO` / `SITE_SELECTION_COVERAGE` for real **Census ACS** demographics and your **own settlement** coverage (paid Marketplace demographics/POI), and the opportunity math is unchanged. The same pattern reuses across the other payments labs.

**Try breaking the verifier:** add a made-up claim (e.g., *"Denver grocery will likely double next year"*) to the draft and watch Step 5 flag it *Needs review*.